In [ ]:
import pymupdf
import json
from pathlib import Path
import pandas as pd

In [ ]:
input_root = Path(r"ALL_PDFS_LPA_EC")
output_root = Path("ALL_PDFS_LPA_EC_JSON")
output_root.mkdir(exist_ok=True)

log_fp = Path("pdf_text_extraction_log.csv")

def is_noise(text: str) -> bool:
    t = text.strip().lower()
    return (len(t) < 3 or t.isdigit() or (t.startswith("page") and len(t) < 20))

def extract_blocks_from_pdf(pdf_path: Path):
    doc = pymupdf.open(pdf_path)
    all_blocks = []

    try:
        for page_num, page in enumerate(doc):
            blocks = page.get_text("blocks")
            blocks = sorted(blocks, key=lambda b: (b[1], b[0]))  #position of the blocks

            for i, b in enumerate(blocks):
                text = b[4].strip()

                if not text:
                    continue

                if is_noise(text):
                    continue

                all_blocks.append({
                    "page": page_num,
                    "block_id": i,
                    "text": text
                })
#for closing the file even if exception gets raised
    finally:
        doc.close()

    return all_blocks


#main loop
pdf_files = sorted(input_root.rglob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs")

log_rows = []

for idx, pdf_path in enumerate(pdf_files, start=1):
    try:
        blocks = extract_blocks_from_pdf(pdf_path)


        relative_pdf = pdf_path.relative_to(input_root)
        json_path = output_root / relative_pdf.with_suffix(".json")
        json_path.parent.mkdir(parents=True, exist_ok=True)

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(blocks, f, indent=2, ensure_ascii=False)

        log_rows.append({
            "json_file": str(json_path),
            "source_file": str(pdf_path),
            "n_blocks": len(blocks),
            "status": "ok",
            "error": ""
        })

    except Exception as e:
        log_rows.append({
            "json_file": "",
            "source_file": str(pdf_path),
            "n_blocks": "",
            "status": "error",
            "error": str(e)
        })

log_df = pd.DataFrame(log_rows)
log_df.to_csv(log_fp, index=False, encoding="utf-8")

print(f"Done. Log saved to: {log_fp}")

Found 3369 PDFs
Done. Log saved to: pdf_text_extraction_log.csv
